# Mathematical Foundations of Boosting Algorithms

> A rigorous, step-by-step derivation written in exam-style notation.  
> Every assumption is justified. Every "why" is answered.

---

## Table of Contents
1. **AdaBoost** — Adaptive Boosting from Exponential Loss
2. **XGBoost** — Extreme Gradient Boosting from Regularised Objectives
3. **GBM** — Gradient Boosting Machine (Friedman's Functional Gradient Descent)
4. **CatBoost** — Categorical Boosting with Ordered Target Statistics
5. **Exam Questions** — Frequently Asked Out-of-Box Questions

---
# 1. AdaBoost (Adaptive Boosting)

## 1.1 The Core Idea

**Question we are solving:** Can we combine many "weak" classifiers (each barely better than random guessing) into one strong classifier?

**Intuition:** Train classifiers sequentially. Each new classifier focuses more on the examples the previous ones got wrong. Finally, take a weighted vote.

---

## 1.2 Problem Setup

Given a training set:

$$\{(x_1, y_1), (x_2, y_2), \ldots, (x_N, y_N)\}$$

where $$ x_i \in \mathcal{X} $$ (feature space) and $$ y_i \in \{-1, +1\} $$ (binary label).

We want to build a strong classifier $$ F(x) $$ as an **additive model**:

$$F(x) = \sum_{m=1}^{M} \alpha_m \, h_m(x)$$

where:
- $$ h_m(x) \in \{-1, +1\} $$ is the $$m$$-th weak classifier (e.g., a decision stump)
- $$ \alpha_m > 0 $$ is the weight (vote) assigned to $$ h_m $$

The final prediction is $$ \hat{y} = \text{sign}(F(x)) $$.

---

### ❓ Why additive model? Why not multiply or stack?

Because addition preserves the **margin** interpretation. The quantity $$ y_i \cdot F(x_i) $$ acts as a confidence measure:
- Large positive → correct and confident
- Large negative → wrong and confident
- Near zero → uncertain

An additive structure lets each new weak learner *incrementally improve* the margin of hard examples.

## 1.3 The Loss Function: Exponential Loss

AdaBoost minimises the **exponential loss**:

$$\mathcal{L} = \sum_{i=1}^{N} \exp\!\big(-y_i \, F(x_i)\big)$$

---

### ❓ Why exponential loss? Why not 0-1 loss or hinge loss?

| Loss Function | Formula | Problem |
|---|---|---|
| 0-1 loss | $$ \mathbb{1}[y \neq \hat{y}] $$ | Non-differentiable, NP-hard to optimise |
| Hinge loss | $$ \max(0, 1 - y \cdot F(x)) $$ | Not differentiable at 1; gives SVM, not boosting |
| **Exponential loss** | $$ e^{-y \cdot F(x)} $$ | Smooth, differentiable, penalises wrong predictions exponentially |

**Key reasons:**
1. It is a smooth, convex upper bound on 0-1 loss (since $$ e^{-z} \geq \mathbb{1}[z \leq 0] $$ for all $$ z $$).
2. It leads to a **closed-form** solution for $$ \alpha_m $$ — no numerical optimisation needed.
3. It naturally gives rise to the re-weighting scheme that defines AdaBoost.

---

### ❓ What is the Bayes-optimal classifier under exponential loss?

The population minimiser of $$ E[e^{-yF(x)} \mid x] $$ is:

$$F^*(x) = \frac{1}{2} \ln \frac{P(y=+1 \mid x)}{P(y=-1 \mid x)}$$

This is **half the log-odds** — the same direction as logistic regression! So exponential loss is a sensible surrogate for classification.

## 1.4 Derivation: Finding the Optimal $$ \alpha_m $$ (Step-by-Step)

We build $$ F(x) $$ stage by stage. At iteration $$ m $$, we already have:

$$F_{m-1}(x) = \sum_{t=1}^{m-1} \alpha_t \, h_t(x)$$

We add one more term:

$$F_m(x) = F_{m-1}(x) + \alpha_m \, h_m(x)$$

---

### Step 1: Write the loss at round $$ m $$

$$\mathcal{L}^{(m)} = \sum_{i=1}^{N} \exp\!\Big(-y_i \, F_m(x_i)\Big)$$

Substitute $$ F_m = F_{m-1} + \alpha_m h_m $$:

$$\mathcal{L}^{(m)} = \sum_{i=1}^{N} \exp\!\Big(-y_i\big(F_{m-1}(x_i) + \alpha_m h_m(x_i)\big)\Big)$$

---

### Step 2: Factor the exponential

Using $$ e^{a+b} = e^a \cdot e^b $$:

$$\mathcal{L}^{(m)} = \sum_{i=1}^{N} \underbrace{\exp\!\big(-y_i F_{m-1}(x_i)\big)}_{\text{Define as } w_i^{(m)}} \cdot \exp\!\big(-\alpha_m y_i h_m(x_i)\big)$$

Define the **sample weight** at round $$ m $$:

$$\boxed{w_i^{(m)} = \exp\!\big(-y_i \, F_{m-1}(x_i)\big)}$$

Note: $$ w_i^{(m)} $$ is fixed (determined by previous rounds) when optimising round $$ m $$.

So:

$$\mathcal{L}^{(m)} = \sum_{i=1}^{N} w_i^{(m)} \cdot \exp\!\big(-\alpha_m \, y_i \, h_m(x_i)\big)$$

---

### ❓ Why is this factorisation powerful?

Because it separates **past history** (captured in $$ w_i^{(m)} $$) from the **current decision** ($$ \alpha_m, h_m $$). This means:
- Points that were hard in previous rounds have large $$ w_i^{(m)} $$
- Points already well-classified have small $$ w_i^{(m)} $$
- The current round naturally focuses on hard examples without explicit re-sampling

---

### Step 3: Exploit the binary nature of $$ y_i $$ and $$ h_m(x_i) $$

Since both $$ y_i \in \{-1,+1\} $$ and $$ h_m(x_i) \in \{-1,+1\} $$, their product:

$$y_i \cdot h_m(x_i) = \begin{cases} +1 & \text{if classified correctly} \\ -1 & \text{if misclassified} \end{cases}$$

Therefore:

$$\exp(-\alpha_m \cdot y_i \cdot h_m(x_i)) = \begin{cases} e^{-\alpha_m} & \text{if } h_m(x_i) = y_i \quad (\text{correct}) \\ e^{+\alpha_m} & \text{if } h_m(x_i) \neq y_i \quad (\text{wrong}) \end{cases}$$

---

### Step 4: Split into correct and incorrect sets

$$\mathcal{L}^{(m)} = e^{-\alpha_m} \!\!\sum_{i:\, h_m(x_i)=y_i} w_i^{(m)} \;+\; e^{+\alpha_m} \!\!\sum_{i:\, h_m(x_i)\neq y_i} w_i^{(m)}$$

Define:
- Total weight: $$ W_m = \sum_{i=1}^{N} w_i^{(m)} $$
- Weighted error rate:

$$\varepsilon_m = \frac{\displaystyle\sum_{i:\, h_m(x_i) \neq y_i} w_i^{(m)}}{\displaystyle\sum_{i=1}^{N} w_i^{(m)}} = \frac{\text{sum of weights of misclassified points}}{\text{sum of all weights}}$$

Then:
- $$ \sum_{i:\, \text{wrong}} w_i^{(m)} = W_m \varepsilon_m $$
- $$ \sum_{i:\, \text{correct}} w_i^{(m)} = W_m (1 - \varepsilon_m) $$

Substitute:

$$\mathcal{L}^{(m)} = W_m \Big[ (1-\varepsilon_m)\,e^{-\alpha_m} + \varepsilon_m \, e^{+\alpha_m} \Big]$$

---

### Step 5: Optimise over $$ \alpha_m $$

Since $$ W_m > 0 $$ is constant w.r.t. $$ \alpha_m $$, we minimise:

$$f(\alpha) = (1-\varepsilon_m)\,e^{-\alpha} + \varepsilon_m \, e^{\alpha}$$

Differentiate w.r.t. $$ \alpha $$:

$$\frac{df}{d\alpha} = -(1-\varepsilon_m)\,e^{-\alpha} + \varepsilon_m \, e^{\alpha}$$

Set to zero:

$$\varepsilon_m \, e^{\alpha} = (1-\varepsilon_m)\,e^{-\alpha}$$

Multiply both sides by $$ e^{\alpha} $$:

$$\varepsilon_m \, e^{2\alpha} = 1-\varepsilon_m$$

Solve:

$$e^{2\alpha} = \frac{1-\varepsilon_m}{\varepsilon_m}$$

Take natural log of both sides:

$$2\alpha = \ln\frac{1-\varepsilon_m}{\varepsilon_m}$$

$$\boxed{\alpha_m = \frac{1}{2}\ln\frac{1-\varepsilon_m}{\varepsilon_m}}$$

---

### Step 6: Verify second-order condition (Is this a minimum?)

$$\frac{d^2 f}{d\alpha^2} = (1-\varepsilon_m)\,e^{-\alpha} + \varepsilon_m \, e^{\alpha} > 0 \quad \forall\,\alpha$$

Since both terms are positive, the critical point is indeed a **minimum**. ✓

---

### ❓ Sanity checks on $$ \alpha_m $$

| Scenario | $$ \varepsilon_m $$ | $$ \alpha_m $$ | Interpretation |
|---|---|---|---|
| Perfect classifier | 0 | $$ +\infty $$ | Infinite trust |
| Useless (random) | 0.5 | 0 | No vote at all |
| Better than random | 0.3 | 0.42 | Moderate positive vote |
| Worse than random | 0.7 | −0.42 | Flip its prediction |

These all make intuitive sense.

## 1.5 Derivation: The Weight Update Rule

We now derive how sample weights evolve from round $$ m $$ to $$ m+1 $$.

---

### Step 1: Express $$ w_i^{(m+1)} $$ in terms of $$ w_i^{(m)} $$

By definition:

$$w_i^{(m+1)} = \exp\!\big(-y_i \, F_m(x_i)\big)$$

Since $$ F_m(x_i) = F_{m-1}(x_i) + \alpha_m h_m(x_i) $$:

$$w_i^{(m+1)} = \exp\!\big(-y_i F_{m-1}(x_i) - y_i \alpha_m h_m(x_i)\big)$$

$$= \exp\!\big(-y_i F_{m-1}(x_i)\big) \cdot \exp\!\big(-\alpha_m y_i h_m(x_i)\big)$$

$$\boxed{w_i^{(m+1)} = w_i^{(m)} \cdot \exp\!\big(-\alpha_m \, y_i \, h_m(x_i)\big)}$$

---

### Step 2: Interpret what happens to each point

**Case 1: Correctly classified** ($$ h_m(x_i) = y_i $$, so $$ y_i h_m(x_i) = +1 $$):

$$w_i^{(m+1)} = w_i^{(m)} \cdot e^{-\alpha_m}$$

Since $$ \alpha_m > 0 $$, we have $$ e^{-\alpha_m} < 1 $$. So weight **decreases**.

**Case 2: Misclassified** ($$ h_m(x_i) \neq y_i $$, so $$ y_i h_m(x_i) = -1 $$):

$$w_i^{(m+1)} = w_i^{(m)} \cdot e^{+\alpha_m}$$

Since $$ e^{+\alpha_m} > 1 $$, weight **increases**.

---

### ❓ Why does this adaptive re-weighting work?

This is the "Ada" (adaptive) in AdaBoost:
- Misclassified points get **upweighted** → next classifier pays more attention to them
- Correctly classified points get **downweighted** → don't waste effort on easy ones
- The magnitude of the shift is controlled by $$ \alpha_m $$: a strong classifier ($$ \alpha_m $$ large) causes a bigger shift

This creates a sequential **curriculum**: each round tackles what previous rounds struggled with.

---

### Step 3: Normalisation

In practice, we normalise weights so they sum to 1:

$$\tilde{w}_i^{(m+1)} = \frac{w_i^{(m+1)}}{\sum_{j=1}^{N} w_j^{(m+1)}}$$

### ❓ Why normalise?

1. **Numerical stability**: Exponentials can grow/shrink to overflow/underflow. Normalisation keeps values bounded.
2. **Probability interpretation**: Normalised weights define a distribution over samples. The weighted error $$ \varepsilon_m $$ becomes an expected error under this distribution.
3. **Equivalence**: Normalisation doesn't change the optimisation because $$ \alpha_m $$ depends only on the *ratio* of weights (through $$ \varepsilon_m $$), and ratios are unchanged by a common scaling factor.

## 1.6 How is $$ h_m $$ Chosen?

Going back to the loss:

$$\mathcal{L}^{(m)} = W_m \Big[ (1-\varepsilon_m)\,e^{-\alpha_m} + \varepsilon_m \, e^{+\alpha_m} \Big]$$

Once we fix $$ h_m $$, we've shown $$ \alpha_m $$ is determined. And the overall loss is **monotonically increasing** in $$ \varepsilon_m $$ (verify: $$ \partial \mathcal{L}/\partial \varepsilon_m > 0 $$).

Therefore, the optimal $$ h_m $$ is the one that **minimises the weighted error rate**:

$$\boxed{h_m = \arg\min_{h \in \mathcal{H}} \sum_{i=1}^{N} w_i^{(m)} \, \mathbb{1}[h(x_i) \neq y_i]}$$

This is just training a weak learner on the weighted dataset!

---

### ❓ Why must it only be "weak" (slightly better than random)?

**Requirement:** $$ \varepsilon_m < 0.5 $$

1. If $$ \varepsilon_m < 0.5 $$, then $$ \alpha_m > 0 $$ and the loss **decreases** at each round.
2. If $$ \varepsilon_m = 0.5 $$, then $$ \alpha_m = 0 $$ and nothing is learned.
3. If $$ \varepsilon_m > 0.5 $$, we flip the classifier.

The "weak learner assumption" guarantees progress. Even a classifier with $$ \varepsilon_m = 0.49 $$ contributes, albeit slowly.

### ❓ What about overfitting? Why doesn't adding more classifiers always overfit?

Empirical and theoretical observations:
1. **Margin theory (Schapire et al., 1998):** AdaBoost increases the *margin* of training examples. A large margin implies good generalisation (analogous to SVM theory).
2. **In practice:** AdaBoost with shallow trees (stumps) is remarkably resistant to overfitting for many rounds, though it CAN overfit eventually, especially on noisy data.

## 1.7 The Complete AdaBoost Algorithm

---

**Input:** Training data $$ \{(x_i, y_i)\}_{i=1}^N $$, number of rounds $$ M $$, weak learner family $$ \mathcal{H} $$

**Initialise:** $$ w_i^{(1)} = \frac{1}{N} $$ for all $$ i = 1,\ldots,N $$

**For** $$ m = 1, 2, \ldots, M $$:

> 1. **Train weak learner** on weighted data:
>    $$h_m = \arg\min_{h \in \mathcal{H}} \sum_{i=1}^{N} w_i^{(m)} \, \mathbb{1}[h(x_i) \neq y_i]$$
> 
> 2. **Compute weighted error:**
>    $$\varepsilon_m = \frac{\sum_{i=1}^{N} w_i^{(m)} \, \mathbb{1}[h_m(x_i) \neq y_i]}{\sum_{i=1}^{N} w_i^{(m)}}$$
> 
> 3. **Compute classifier weight:**
>    $$\alpha_m = \frac{1}{2} \ln\frac{1 - \varepsilon_m}{\varepsilon_m}$$
> 
> 4. **Update sample weights:**
>    $$w_i^{(m+1)} = w_i^{(m)} \cdot \exp\!\big(-\alpha_m \, y_i \, h_m(x_i)\big)$$
> 
> 5. **Normalise:** $$ w_i^{(m+1)} \leftarrow \frac{w_i^{(m+1)}}{\sum_j w_j^{(m+1)}} $$

**Output:** Final classifier:

$$\boxed{F(x) = \text{sign}\!\left(\sum_{m=1}^{M} \alpha_m \, h_m(x)\right)}$$

---

### ❓ Why initialise with uniform weights $$ 1/N $$?

Before seeing any classifier, we have no reason to prefer any training point over another. Uniform weights = **maximum ignorance** (maximum entropy prior). It's the only unbiased starting point.

---

## 1.8 Training Error Bound

One can prove (Freund & Schapire, 1997):

$$\text{Training error} \leq \prod_{m=1}^{M} 2\sqrt{\varepsilon_m(1-\varepsilon_m)}$$

Since $$ \varepsilon_m < 0.5 $$, each factor $$ 2\sqrt{\varepsilon_m(1-\varepsilon_m)} < 1 $$.

So the training error **decreases exponentially** with the number of rounds!

Define the "edge" $$ \gamma_m = \frac{1}{2} - \varepsilon_m > 0 $$. Then:

$$2\sqrt{\varepsilon_m(1-\varepsilon_m)} = \sqrt{1 - 4\gamma_m^2} \leq e^{-2\gamma_m^2}$$

So:

$$\text{Training error} \leq \exp\!\left(-2\sum_{m=1}^{M} \gamma_m^2\right)$$

Even if each weak learner has a tiny edge $$ \gamma $$, after $$ M $$ rounds the error is at most $$ e^{-2M\gamma^2} \to 0 $$.

### ❓ What does this bound tell us practically?

- You don't need strong classifiers. Decision stumps ($$ \gamma \approx 0.1 $$) suffice.
- More rounds = lower training error (guaranteed, not just empirical).
- The convergence rate depends on how good your weak learners are (through $$ \gamma $$).

---
---
# 2. XGBoost (Extreme Gradient Boosting)

## 2.1 Motivation: What Problem Does XGBoost Solve?

AdaBoost uses exponential loss and binary classifiers. **XGBoost generalises this:**

| Feature | AdaBoost | XGBoost |
|---|---|---|
| Loss function | Exponential only | **Any** differentiable loss |
| Base learner output | $$ \{-1, +1\} $$ | Real-valued (regression trees) |
| Regularisation | None (implicit via weak learners) | **Explicit** penalty on tree complexity |
| Optimisation | Exact closed-form | Second-order Taylor approximation |
| Task | Binary classification | Classification, regression, ranking, ... |

---

## 2.2 Problem Setup

Given training data $$ \{(x_i, y_i)\}_{i=1}^N $$.

Build a prediction as a sum of $$ K $$ regression trees:

$$\hat{y}_i = F(x_i) = \sum_{k=1}^{K} f_k(x_i)$$

where each $$ f_k $$ is a **regression tree** that maps $$ x \to \mathbb{R} $$.

A tree $$ f $$ is characterised by:
- A **structure** $$ q: \mathbb{R}^d \to \{1, 2, \ldots, T\} $$ that maps input to leaf index
- **Leaf weights** $$ \mathbf{w} = (w_1, w_2, \ldots, w_T) \in \mathbb{R}^T $$

So $$ f(x) = w_{q(x)} $$ (the output is the weight of the leaf that $$ x $$ lands in).

---

### ❓ Why regression trees instead of classifiers?

Regression trees output **real numbers**, not just labels. This means:
1. We can do regression, not just classification
2. The output is a "score" that can be optimised continuously
3. We can apply gradient-based optimisation (requires continuous outputs)
4. Multiple classes are handled by one tree per class (or one tree outputting a vector)

## 2.3 The Objective Function

XGBoost minimises a **regularised objective**:

$$\text{Obj} = \underbrace{\sum_{i=1}^{N} L\big(y_i,\, \hat{y}_i\big)}_{\text{Training loss}} + \underbrace{\sum_{k=1}^{K} \Omega(f_k)}_{\text{Regularisation}}$$

where $$ L $$ is any twice-differentiable loss and $$ \Omega $$ penalises tree complexity.

---

### The Regularisation Term

For a single tree with $$ T $$ leaves and leaf weights $$ \mathbf{w} $$:

$$\boxed{\Omega(f) = \gamma \, T + \frac{1}{2}\lambda \sum_{j=1}^{T} w_j^2}$$

where:
- $$ \gamma $$ = penalty per leaf (controls tree depth / number of leaves)
- $$ \lambda $$ = L2 penalty on leaf weights (shrinks predictions toward zero)

---

### ❓ Why this specific form of regularisation?

**Why $$ \gamma T $$ (leaf count penalty)?**
- Each leaf represents a partition of the feature space. More leaves = more complex model.
- $$ \gamma $$ acts like a "minimum gain" threshold: a split is only worth it if the gain exceeds $$ \gamma $$.
- This is equivalent to **pre-pruning** — but derived from the objective, not a heuristic.

**Why $$ \frac{1}{2}\lambda \|\mathbf{w}\|^2 $$ (L2 on leaf weights)?**
- Large leaf weights mean extreme predictions. L2 shrinks them toward zero.
- This is analogous to **ridge regression** at the leaf level.
- Prevents any single leaf from dominating the prediction.
- The $$ \frac{1}{2} $$ is a convenience (cancels with the 2 from the derivative).

**Why NOT L1 (like Lasso)?**
- L1 ($$ |w_j| $$) could also work and would encourage **sparse** leaf weights (some leaves output exactly 0).
- But L2 gives a **closed-form** optimal $$ w_j $$ — which is crucial for the derivation below.
- XGBoost also supports L1 in practice (via an additional $$ \alpha \sum |w_j| $$ term), but the core math uses L2.

---

### ❓ Why regularise at all? AdaBoost didn't need it.

AdaBoost's base learners are **stumps** (1-2 leaves). Their capacity is so limited that overfitting is controlled implicitly.

XGBoost uses deeper trees (typically depth 3–8). Without regularisation:
- Trees can memorise noise
- Leaf weights can be extreme when few samples reach a leaf
- The model has too many effective parameters

Regularisation is the principled way to control capacity when base learners are powerful.

## 2.4 Additive Training (Greedy Stage-wise Approach)

Like AdaBoost, we build the model one tree at a time.

At step $$ t $$, the prediction is:

$$\hat{y}_i^{(t)} = \hat{y}_i^{(t-1)} + f_t(x_i)$$

The objective at step $$ t $$:

$$\text{Obj}^{(t)} = \sum_{i=1}^{N} L\big(y_i,\; \hat{y}_i^{(t-1)} + f_t(x_i)\big) + \Omega(f_t) + \text{const}$$

where "const" includes the regularisation of previous trees (already fixed).

---

### ❓ Why greedy (one tree at a time)? Why not optimise all trees jointly?

1. **Computational tractability:** Joint optimisation over $$ K $$ tree structures is combinatorially explosive.
2. **The additive structure decomposes:** Once $$ f_1, \ldots, f_{t-1} $$ are fixed, only $$ f_t $$ is unknown.
3. **Gradient boosting theory (Friedman, 2001):** Greedy stage-wise addition is a functional gradient descent in function space, with convergence guarantees.

---

### The Key Challenge

For **general** $$ L $$, the expression $$ L(y_i, \hat{y}_i^{(t-1)} + f_t(x_i)) $$ is hard to optimise directly over tree structures.

XGBoost's solution: **Approximate** the loss with a Taylor expansion.

## 2.5 The Taylor Expansion Trick (Step-by-Step)

This is the **core mathematical insight** of XGBoost.

---

### Step 1: Recall the Taylor expansion

For a scalar function $$ g(x) $$ expanded around point $$ a $$:

$$g(a + \Delta) \approx g(a) + g'(a)\,\Delta + \frac{1}{2}g''(a)\,\Delta^2$$

Here:
- $$ a = \hat{y}_i^{(t-1)} $$ (previous prediction, fixed)
- $$ \Delta = f_t(x_i) $$ (the new tree's output, what we're optimising)
- $$ g(\cdot) = L(y_i, \cdot) $$ (loss as a function of the prediction)

---

### Step 2: Define the gradient and Hessian

Let:

$$g_i = \frac{\partial L(y_i, \hat{y})}{\partial \hat{y}} \Bigg|_{\hat{y} = \hat{y}_i^{(t-1)}}$$

$$h_i = \frac{\partial^2 L(y_i, \hat{y})}{\partial \hat{y}^2} \Bigg|_{\hat{y} = \hat{y}_i^{(t-1)}}$$

These are computed from the **previous round's predictions** — they are constants when optimising $$ f_t $$.

---

### Step 3: Apply the expansion

$$L\big(y_i,\; \hat{y}_i^{(t-1)} + f_t(x_i)\big) \approx L\big(y_i,\; \hat{y}_i^{(t-1)}\big) + g_i \, f_t(x_i) + \frac{1}{2} h_i \, f_t(x_i)^2$$

---

### Step 4: Substitute into the objective

$$\text{Obj}^{(t)} \approx \sum_{i=1}^{N} \left[ L\big(y_i, \hat{y}_i^{(t-1)}\big) + g_i \, f_t(x_i) + \frac{1}{2} h_i \, f_t(x_i)^2 \right] + \Omega(f_t) + \text{const}$$

The term $$ L(y_i, \hat{y}_i^{(t-1)}) $$ is a constant (doesn't depend on $$ f_t $$). Drop it:

$$\boxed{\widetilde{\text{Obj}}^{(t)} = \sum_{i=1}^{N} \left[ g_i \, f_t(x_i) + \frac{1}{2} h_i \, f_t(x_i)^2 \right] + \Omega(f_t)}$$

This is the **simplified objective** that XGBoost actually optimises.

---

### ❓ Why second-order (Taylor) and not just first-order (gradient)?

**First-order only** (standard gradient boosting):

$$\text{Obj}^{(t)} \approx \sum_i g_i \, f_t(x_i) + \text{const}$$

This just says "go in the negative gradient direction" but gives **no information about step size**. You'd need a separate line search or learning rate.

**Second-order** (XGBoost):

The $$ h_i $$ term provides **curvature information** — it tells us *how far* to step:
- Large $$ h_i $$ → the loss is sharply curved → take a smaller step
- Small $$ h_i $$ → the loss is flat → can take a bigger step

This is exactly the Newton's method intuition: the optimal step is $$ -g/h $$, not just $$ -g $$.

**Benefits:**
1. Faster convergence (Newton > gradient descent)
2. **Closed-form** optimal leaf weights (no line search needed)
3. Works for any loss — just plug in $$ g_i, h_i $$

---

### ❓ Is the approximation accurate?

Yes, because:
1. We use a **small learning rate** ($$ \eta \approx 0.1 $$), so $$ f_t(x_i) $$ is small
2. For small $$ \Delta $$, the second-order Taylor expansion is excellent
3. We're doing many rounds, each with a small step — errors accumulate slowly
4. For squared loss, the approximation is actually **exact** (since squared loss is quadratic):

$$L = (y_i - \hat{y})^2 \Rightarrow g_i = -2(y_i - \hat{y}_i^{(t-1)}),\; h_i = 2$$

---

### ❓ What are $$ g_i $$ and $$ h_i $$ for common losses?

| Loss | $$ L(y, \hat{y}) $$ | $$ g_i $$ | $$ h_i $$ |
|---|---|---|---|
| Squared error | $$ \frac{1}{2}(y-\hat{y})^2 $$ | $$ \hat{y}_i^{(t-1)} - y_i $$ | $$ 1 $$ |
| Logistic | $$ y\ln p + (1-y)\ln(1-p) $$ where $$ p = \sigma(\hat{y}) $$ | $$ p_i - y_i $$ | $$ p_i(1-p_i) $$ |
| Exponential | $$ e^{-y\hat{y}} $$ | $$ -y_i e^{-y_i \hat{y}_i^{(t-1)}} $$ | $$ e^{-y_i \hat{y}_i^{(t-1)}} $$ |

## 2.6 Derivation: Optimal Leaf Weights (Step-by-Step)

Now we derive the **closed-form optimal leaf weights** for a given tree structure.

---

### Step 1: Re-write $$ f_t(x_i) $$ in terms of leaves

Recall: tree $$ f_t $$ maps each $$ x_i $$ to the weight of the leaf it lands in:

$$f_t(x_i) = w_{q(x_i)}$$

where $$ q(x_i) \in \{1, \ldots, T\} $$ is the leaf index for point $$ x_i $$.

---

### Step 2: Group samples by leaf

Define $$ I_j = \{i : q(x_i) = j\} $$ = set of sample indices that fall into leaf $$ j $$.

Then:

$$\sum_{i=1}^{N} g_i \, f_t(x_i) = \sum_{i=1}^{N} g_i \, w_{q(x_i)} = \sum_{j=1}^{T} \left(\sum_{i \in I_j} g_i\right) w_j$$

Similarly:

$$\sum_{i=1}^{N} h_i \, f_t(x_i)^2 = \sum_{j=1}^{T} \left(\sum_{i \in I_j} h_i\right) w_j^2$$

Define the **per-leaf gradient and Hessian sums**:

$$G_j = \sum_{i \in I_j} g_i \qquad H_j = \sum_{i \in I_j} h_i$$

---

### Step 3: Substitute into the objective

Expand the regularisation:

$$\Omega(f_t) = \gamma T + \frac{1}{2}\lambda \sum_{j=1}^{T} w_j^2$$

The full simplified objective becomes:

$$\widetilde{\text{Obj}}^{(t)} = \sum_{j=1}^{T} \left[ G_j \, w_j + \frac{1}{2} H_j \, w_j^2 \right] + \gamma T + \frac{1}{2}\lambda \sum_{j=1}^{T} w_j^2$$

Combine the $$ w_j^2 $$ terms:

$$\boxed{\widetilde{\text{Obj}}^{(t)} = \sum_{j=1}^{T} \left[ G_j \, w_j + \frac{1}{2}(H_j + \lambda)\, w_j^2 \right] + \gamma T}$$

---

### ❓ Why is this form beautiful?

The objective **decomposes over leaves**! Each leaf $$ j $$ contributes independently:

$$\text{Obj}_j(w_j) = G_j \, w_j + \frac{1}{2}(H_j + \lambda)\, w_j^2$$

This is a simple quadratic in $$ w_j $$. We can optimise each leaf separately.

---

### Step 4: Optimise each leaf weight

For leaf $$ j $$, take derivative and set to zero:

$$\frac{\partial \text{Obj}_j}{\partial w_j} = G_j + (H_j + \lambda)\, w_j = 0$$

Solve:

$$\boxed{w_j^* = -\frac{G_j}{H_j + \lambda}}$$

This is the **optimal prediction** for leaf $$ j $$.

---

### ❓ Interpreting $$ w_j^* = -G_j / (H_j + \lambda) $$

**It's Newton's step!**

- $$ G_j = \sum_{i \in I_j} g_i $$ is the total gradient (direction of steepest ascent of loss)
- $$ H_j = \sum_{i \in I_j} h_i $$ is the total curvature
- The step is $$ -\text{gradient}/\text{curvature} $$ = Newton's method
- $$ \lambda $$ adds damping: prevents extreme steps when $$ H_j $$ is small (few samples or flat loss)

**Special case (squared loss):** $$ g_i = \hat{y}_i - y_i $$, $$ h_i = 1 $$, $$ \lambda = 0 $$:

$$w_j^* = -\frac{\sum_{i \in I_j}(\hat{y}_i - y_i)}{|I_j|} = \text{mean residual in leaf } j$$

Exactly what you'd expect!

---

### Step 5: Optimal objective value (quality score)

Substitute $$ w_j^* $$ back:

$$\text{Obj}_j^* = G_j \cdot \left(-\frac{G_j}{H_j+\lambda}\right) + \frac{1}{2}(H_j+\lambda)\left(-\frac{G_j}{H_j+\lambda}\right)^2$$

$$= -\frac{G_j^2}{H_j+\lambda} + \frac{1}{2} \cdot \frac{G_j^2}{H_j+\lambda}$$

$$= -\frac{1}{2} \cdot \frac{G_j^2}{H_j+\lambda}$$

Total optimal objective:

$$\boxed{\widetilde{\text{Obj}}^* = -\frac{1}{2} \sum_{j=1}^{T} \frac{G_j^2}{H_j + \lambda} + \gamma T}$$

This is the **quality score** of a given tree structure. Lower = better.

## 2.7 Derivation: The Split Gain Formula

Now the key question: **How do we decide where to split?**

Given a leaf node with samples $$ I $$, we consider splitting into left ($$ I_L $$) and right ($$ I_R $$) such that $$ I = I_L \cup I_R $$.

---

### Step 1: Score before the split

Before splitting, the leaf contributes:

$$\text{Score}_{\text{before}} = -\frac{1}{2} \cdot \frac{(G_L + G_R)^2}{H_L + H_R + \lambda} + \gamma$$

(One leaf, total gradient $$ G_L + G_R $$, total Hessian $$ H_L + H_R $$)

---

### Step 2: Score after the split

After splitting, we have two leaves:

$$\text{Score}_{\text{after}} = -\frac{1}{2} \cdot \frac{G_L^2}{H_L + \lambda} - \frac{1}{2} \cdot \frac{G_R^2}{H_R + \lambda} + 2\gamma$$

(Two leaves, each with their own score, plus penalty for the extra leaf)

---

### Step 3: Compute the gain

$$\text{Gain} = \text{Score}_{\text{before}} - \text{Score}_{\text{after}}$$

But since **lower score is better**, the improvement from splitting is:

$$\text{Gain} = \text{Score}_{\text{before}} - \text{Score}_{\text{after}}$$

Substitute (note: scores are negative, so "before minus after" when before is worse means after is more negative):

$$\text{Gain} = \left[-\frac{1}{2}\frac{(G_L+G_R)^2}{H_L+H_R+\lambda} + \gamma\right] - \left[-\frac{1}{2}\frac{G_L^2}{H_L+\lambda} - \frac{1}{2}\frac{G_R^2}{H_R+\lambda} + 2\gamma\right]$$

Simplify:

$$\boxed{\text{Gain} = \frac{1}{2}\left[\frac{G_L^2}{H_L + \lambda} + \frac{G_R^2}{H_R + \lambda} - \frac{(G_L + G_R)^2}{H_L + H_R + \lambda}\right] - \gamma}$$

---

### ❓ Interpreting the Gain formula term by term

$$\text{Gain} = \underbrace{\frac{1}{2}\frac{G_L^2}{H_L + \lambda}}_{\text{Score of left child}} + \underbrace{\frac{1}{2}\frac{G_R^2}{H_R + \lambda}}_{\text{Score of right child}} - \underbrace{\frac{1}{2}\frac{(G_L+G_R)^2}{H_L+H_R+\lambda}}_{\text{Score of no-split}} - \underbrace{\gamma}_{\text{Cost of new leaf}}$$

**In words:** The gain measures how much the objective *improves* by splitting, minus the *penalty* for adding one more leaf.

---

### ❓ When is a split worthwhile?

Only when $$ \text{Gain} > 0 $$, i.e.:

$$\frac{G_L^2}{H_L + \lambda} + \frac{G_R^2}{H_R + \lambda} - \frac{(G_L+G_R)^2}{H_L+H_R+\lambda} > 2\gamma$$

So $$ \gamma $$ acts as a **minimum improvement threshold** for splitting. Larger $$ \gamma $$ = fewer splits = simpler trees.

---

### ❓ Why is $$ \frac{G^2}{H + \lambda} $$ the right "impurity" measure?

This is analogous to variance reduction in CART, but **generalised for any loss**:

- For squared loss ($$ h_i = 1 $$): $$ G_j^2 / H_j = (\sum \text{residuals})^2 / n_j $$ which is proportional to the reduction in MSE
- For logistic loss: the formula automatically adapts to the curvature of the logistic function
- The term $$ G^2/H $$ measures how much the loss can be reduced in that region — the "potential" for improvement

This unified formula handles regression, classification, and ranking with the same code — just change $$ g_i, h_i $$.

## 2.8 Shrinkage (Learning Rate)

In practice, XGBoost uses a **learning rate** $$ \eta \in (0, 1] $$:

$$\hat{y}_i^{(t)} = \hat{y}_i^{(t-1)} + \eta \cdot f_t(x_i)$$

Typically $$ \eta \in [0.01, 0.3] $$.

---

### ❓ Why not just use the optimal $$ f_t $$ directly (i.e., $$ \eta = 1 $$)?

1. **Regularisation effect:** Small $$ \eta $$ prevents any single tree from having too much influence. It's like a "conservation" principle — don't be too greedy.

2. **Statistical averaging:** With small $$ \eta $$, you need more trees ($$ K $$) to reach the same training loss. But each tree sees slightly different residuals, creating a **diverse ensemble** — analogous to how slow learning in SGD finds flatter minima.

3. **The bias-variance tradeoff:** 
   - $$ \eta = 1 $$: Fast convergence, fewer trees needed, but each tree overfits to current residuals
   - $$ \eta = 0.1 $$: Slower convergence, more trees, but better generalisation

4. **Theoretical justification (Friedman, 2001):** Shrinkage in boosting is analogous to a small step size in gradient descent. It doesn't change the direction but controls how far we go, preventing overshooting.

---

### ❓ How does $$ \eta $$ interact with the number of trees $$ K $$?

Roughly:

$$\text{Required } K \propto \frac{1}{\eta}$$

Small $$ \eta $$ + large $$ K $$ generally outperforms large $$ \eta $$ + small $$ K $$, at the cost of more computation.

## 2.9 The Complete XGBoost Algorithm

---

**Input:** Training data $$ \{(x_i, y_i)\}_{i=1}^N $$, loss function $$ L $$, hyperparameters $$ (\eta, \gamma, \lambda, K, \text{max\_depth}) $$

**Initialise:** $$ \hat{y}_i^{(0)} = 0 $$ for all $$ i $$ (or a constant like the mean of $$ y $$)

**For** $$ t = 1, 2, \ldots, K $$:

> 1. **Compute gradients and Hessians** for all samples:
>    $$g_i = \frac{\partial L(y_i, \hat{y})}{\partial \hat{y}}\Bigg|_{\hat{y}=\hat{y}_i^{(t-1)}} \qquad h_i = \frac{\partial^2 L(y_i, \hat{y})}{\partial \hat{y}^2}\Bigg|_{\hat{y}=\hat{y}_i^{(t-1)}}$$
>
> 2. **Build a tree** $$ f_t $$ by greedily finding splits:
>    - For each candidate split, compute:
>      $$\text{Gain} = \frac{1}{2}\left[\frac{G_L^2}{H_L+\lambda} + \frac{G_R^2}{H_R+\lambda} - \frac{(G_L+G_R)^2}{H_L+H_R+\lambda}\right] - \gamma$$
>    - Choose the split with maximum Gain
>    - Stop splitting when Gain $$ \leq 0 $$ or max depth reached
>
> 3. **Assign leaf weights:**
>    $$w_j^* = -\frac{G_j}{H_j + \lambda} \quad \text{for each leaf } j$$
>
> 4. **Update predictions with shrinkage:**
>    $$\hat{y}_i^{(t)} = \hat{y}_i^{(t-1)} + \eta \cdot f_t(x_i)$$

**Output:** Final prediction:

$$\boxed{\hat{y}_i = \hat{y}_i^{(0)} + \eta \sum_{t=1}^{K} f_t(x_i)}$$

For classification, apply sigmoid: $$ P(y=1 \mid x) = \sigma(\hat{y}) = \frac{1}{1+e^{-\hat{y}}} $$

---

### ❓ How does XGBoost find splits efficiently in practice?

**Exact greedy:** Sort samples by each feature, scan all possible split points. Complexity: $$ O(N \cdot d \cdot K) $$ where $$ d $$ is number of features.

**Approximate (histogram-based):** Bucket continuous features into quantiles. Only evaluate split points at bucket boundaries. This is what makes XGBoost scale to large datasets:
- Sort once, scan quantiles: $$ O(N + q \cdot d) $$ where $$ q $$ is number of quantiles
- Histogram subtraction trick: compute right child = parent - left child (saves 50% work)

## 2.10 Connecting XGBoost Back to AdaBoost

XGBoost with **exponential loss** and **depth-1 trees** reduces to AdaBoost! Let's verify:

Set $$ L(y, \hat{y}) = e^{-y\hat{y}} $$ with $$ y \in \{-1,+1\} $$:

$$g_i = -y_i \, e^{-y_i \hat{y}_i^{(t-1)}} \qquad h_i = e^{-y_i \hat{y}_i^{(t-1)}}$$

Note that $$ h_i = |g_i| / |y_i| = |g_i| $$ (since $$ |y_i|=1 $$).

The sample weights in AdaBoost were $$ w_i = e^{-y_i F_{t-1}(x_i)} = h_i $$.

The optimal leaf weight becomes:

$$w_j^* = -\frac{\sum_{i \in I_j} g_i}{\sum_{i \in I_j} h_i + \lambda} = \frac{\sum_{i \in I_j} y_i \, h_i}{\sum_{i \in I_j} h_i + \lambda}$$

With $$ \lambda = 0 $$ and depth-1 trees (two leaves split by one feature), this recovers the AdaBoost formula.

---

### The Hierarchy of Boosting Methods

$$\text{AdaBoost} \subset \text{Gradient Boosting} \subset \text{XGBoost}$$

| Method | Loss | Approximation | Regularisation |
|---|---|---|---|
| AdaBoost | Exponential | Exact | None |
| Gradient Boosting (Friedman) | Any differentiable | First-order | Learning rate only |
| **XGBoost** | Any twice-differentiable | **Second-order** | $$ \gamma T + \frac{1}{2}\lambda\|w\|^2 $$ |

---

## 2.11 Summary: What Makes XGBoost "Extreme"?

1. **Second-order optimisation** → faster convergence than gradient boosting
2. **Explicit regularisation** → better generalisation
3. **Approximate split finding** (weighted quantile sketch) → scalability
4. **Sparsity awareness** → handles missing values natively
5. **Column block structure** → cache-efficient parallel tree building
6. **Out-of-core computation** → handles data larger than RAM

The "extreme" refers to pushing the engineering and mathematical boundaries of gradient boosting to their limits.

---
---
# 3. Gradient Boosting Machine (GBM)

## 3.1 The Core Insight: Gradient Descent in Function Space

Friedman (2001) asked: **What if we treat boosting as gradient descent, but in the space of functions instead of parameters?**

In classical gradient descent (parameter space):
- We have parameters $$ \theta \in \mathbb{R}^p $$
- Update: $$ \theta^{(t)} = \theta^{(t-1)} - \eta \nabla_{\theta} L $$

In **functional** gradient descent:
- We have a function $$ F: \mathcal{X} \to \mathbb{R} $$
- Update: $$ F^{(t)}(x) = F^{(t-1)}(x) - \eta \cdot \nabla_F L $$

But what does "gradient of a loss w.r.t. a function" mean?

---

### The Functional Gradient

Given loss $$ \mathcal{L}(F) = \sum_{i=1}^{N} L(y_i, F(x_i)) $$, the functional gradient at point $$ x_i $$ is:

$$\left[\nabla_F \mathcal{L}\right](x_i) = \frac{\partial L(y_i, F(x_i))}{\partial F(x_i)}$$

This is just the partial derivative of the loss w.r.t. the prediction at each data point!

---

### ❓ Why is this "in function space" and not just regular gradient descent?

Because:
1. We're not optimising a fixed set of parameters. We're optimising the **function values** $$ F(x_1), F(x_2), \ldots, F(x_N) $$ — one degree of freedom per data point.
2. The ideal update would be: set $$ F(x_i) \leftarrow F(x_i) - \eta \cdot g_i $$ at each training point.
3. But we need $$ F $$ to generalise to new points! So we **approximate** the negative gradient with a base learner (e.g., a tree) that can predict at unseen $$ x $$.

This is the bridge between optimisation theory and machine learning.

---

## 3.2 Problem Setup

Same as XGBoost but **without explicit regularisation in the objective**:

$$\text{Obj} = \sum_{i=1}^{N} L(y_i, F(x_i))$$

where $$ L $$ is any differentiable loss function.

Model:

$$F(x) = F_0(x) + \sum_{m=1}^{M} \eta \cdot h_m(x)$$

where $$ F_0 $$ is an initial guess, $$ h_m $$ are base learners (regression trees), and $$ \eta $$ is the learning rate.

---

### ❓ What is $$ F_0 $$? Why do we need it?

$$ F_0 $$ is the **best constant prediction** — the value that minimises the loss before any trees are added:

$$F_0 = \arg\min_{c} \sum_{i=1}^{N} L(y_i, c)$$

Examples:
- Squared loss: $$ F_0 = \bar{y} $$ (the mean)
- Absolute loss: $$ F_0 = \text{median}(y) $$
- Log-loss: $$ F_0 = \log\frac{\bar{y}}{1 - \bar{y}} $$ (log-odds of the base rate)

Starting from $$ F_0 $$ instead of 0 ensures the first tree corrects deviations from a reasonable baseline, not from scratch.

## 3.3 Derivation: Why Fit to Negative Gradients ("Pseudo-Residuals")

---

### Step 1: The ideal update

At iteration $$ m $$, we want to find a function $$ h_m(x) $$ such that adding it reduces the loss:

$$F_m(x) = F_{m-1}(x) + \eta \cdot h_m(x)$$

The steepest descent direction in function space is the **negative gradient**:

$$-\frac{\partial L(y_i, F(x_i))}{\partial F(x_i)} \Bigg|_{F = F_{m-1}}$$

Define the **pseudo-residual** for sample $$ i $$:

$$\boxed{r_{im} = -\frac{\partial L(y_i, F(x_i))}{\partial F(x_i)} \Bigg|_{F = F_{m-1}}}$$

---

### Step 2: Fit a base learner to pseudo-residuals

We can only evaluate the gradient at training points $$ x_1, \ldots, x_N $$. To generalise to new points, we train a base learner:

$$h_m = \arg\min_{h \in \mathcal{H}} \sum_{i=1}^{N} (r_{im} - h(x_i))^2$$

This fits a regression tree to the pseudo-residuals using least squares.

---

### ❓ Why are they called "pseudo-residuals"?

For **squared loss** $$ L = \frac{1}{2}(y - F(x))^2 $$:

$$r_{im} = -\frac{\partial}{\partial F}\frac{1}{2}(y_i - F)^2 = y_i - F_{m-1}(x_i)$$

These are the actual residuals! So for squared loss, GBM literally fits trees to residuals.

For **other losses**, $$ r_{im} $$ is NOT a true residual, but it plays the same role — it tells each sample "in which direction and how much the prediction should change." Hence "pseudo-residual."

---

### ❓ What do pseudo-residuals look like for other losses?

| Loss | $$ L(y, F) $$ | Pseudo-residual $$ r_{im} $$ |
|---|---|---|
| Squared error | $$ \frac{1}{2}(y-F)^2 $$ | $$ y_i - F_{m-1}(x_i) $$ |
| Absolute error | $$ |y - F| $$ | $$ \text{sign}(y_i - F_{m-1}(x_i)) $$ |
| Huber loss | Combination | Clipped residual |
| Log-loss (classification) | $$ -y\ln p - (1-y)\ln(1-p) $$ | $$ y_i - p_i $$ where $$ p_i = \sigma(F_{m-1}(x_i)) $$ |

Notice for log-loss: the pseudo-residual $$ y_i - p_i $$ is the "gap" between the true label and the predicted probability. Intuitively correct!

---

### ❓ Why fit with least squares even when the loss isn't squared?

Great question! We use squared loss to fit $$ h_m $$ to $$ r_{im} $$ because:

1. **The pseudo-residual is a direction, not a target:** We want $$ h_m(x_i) $$ to be a good function approximation of the gradient vector. Least squares is the natural way to project a vector onto a function class.

2. **Computational efficiency:** Trees with squared loss splits are fast and well-understood.

3. **It's sufficient:** We don't need $$ h_m $$ to perfectly match $$ r_{im} $$ — we only need it to be positively correlated with the gradient direction (i.e., a descent direction). Least squares guarantees this.

4. **The step size handles the rest:** Even if the magnitude is wrong, the line search or learning rate corrects it.

## 3.4 Derivation: Per-Leaf Line Search (Optimal Step Size)

Friedman's key refinement: instead of using a single step size for the entire tree, use a **different step size for each leaf region**.

---

### Step 1: Tree partitions the space into regions

Let tree $$ h_m $$ have $$ J $$ terminal regions (leaves) $$ R_{1m}, R_{2m}, \ldots, R_{Jm} $$.

The tree output can be written as:

$$h_m(x) = \sum_{j=1}^{J} b_{jm} \, \mathbb{1}[x \in R_{jm}]$$

where $$ b_{jm} $$ is the raw prediction from fitting to pseudo-residuals.

---

### Step 2: Replace $$ b_{jm} $$ with optimal per-leaf coefficients

Instead of using $$ b_{jm} $$, find the optimal multiplier $$ \rho_{jm} $$ for each leaf:

$$F_m(x) = F_{m-1}(x) + \eta \sum_{j=1}^{J} \rho_{jm} \, \mathbb{1}[x \in R_{jm}]$$

The optimal $$ \rho_{jm} $$ solves:

$$\boxed{\rho_{jm} = \arg\min_{\rho} \sum_{x_i \in R_{jm}} L\big(y_i, \, F_{m-1}(x_i) + \eta \cdot \rho\big)}$$

This is a **one-dimensional optimisation** per leaf.

---

### Step 3: Closed-form solutions for common losses

**Squared loss:**

$$\rho_{jm} = \frac{\sum_{x_i \in R_{jm}} (y_i - F_{m-1}(x_i))}{|R_{jm}|} = \text{mean residual in leaf } j$$

**Absolute loss (LAD regression):**

$$\rho_{jm} = \text{median}_{x_i \in R_{jm}} (y_i - F_{m-1}(x_i))$$

**Log-loss (classification):**

$$\rho_{jm} = \frac{\sum_{x_i \in R_{jm}} r_{im}}{\sum_{x_i \in R_{jm}} |r_{im}|(1 - |r_{im}|)}$$

which is the Newton-Raphson step for logistic regression restricted to leaf $$ j $$.

---

### ❓ Why per-leaf step sizes instead of one global step?

Consider two leaves:
- Leaf A: contains 100 samples, all with small residuals
- Leaf B: contains 5 samples, all with large residuals

A global step would over-shoot for Leaf A or under-shoot for Leaf B. Per-leaf optimisation gives each region its **own optimal correction**.

This is why GBM with per-leaf line search is already a step toward XGBoost's leaf weights $$ w_j^* = -G_j/(H_j + \lambda) $$ — without the regularisation.

---

### ❓ How does this relate to XGBoost's leaf weights?

XGBoost's $$ w_j^* = -G_j/(H_j + \lambda) $$ is exactly the Newton step for the per-leaf optimisation above, with L2 regularisation:

- $$ G_j = \sum_{i \in R_j} g_i $$ = sum of gradients (first-order)
- $$ H_j = \sum_{i \in R_j} h_i $$ = sum of Hessians (second-order curvature)
- With $$ \lambda = 0 $$: $$ w_j^* = -G_j/H_j $$ = Newton's step = same as GBM's line search with second-order expansion

So **GBM with line search \u2248 XGBoost without regularisation** (for smooth losses where line search finds the Newton point).

## 3.5 The Complete GBM Algorithm

---

**Input:** Training data $$ \{(x_i, y_i)\}_{i=1}^N $$, differentiable loss $$ L $$, number of iterations $$ M $$, learning rate $$ \eta $$, tree depth

**Step 0: Initialise** with the best constant:

$$F_0(x) = \arg\min_c \sum_{i=1}^{N} L(y_i, c)$$

**For** $$ m = 1, 2, \ldots, M $$:

> 1. **Compute pseudo-residuals** for all $$ i = 1, \ldots, N $$:
>    $$r_{im} = -\frac{\partial L(y_i, F(x_i))}{\partial F(x_i)} \Bigg|_{F = F_{m-1}}$$
>
> 2. **Fit a regression tree** to $$ \{(x_i, r_{im})\}_{i=1}^N $$, giving terminal regions $$ R_{1m}, \ldots, R_{Jm} $$
>
> 3. **Compute optimal per-leaf values** (line search):
>    $$\rho_{jm} = \arg\min_\rho \sum_{x_i \in R_{jm}} L(y_i, F_{m-1}(x_i) + \rho) \quad \text{for } j = 1, \ldots, J$$
>
> 4. **Update the model:**
>    $$F_m(x) = F_{m-1}(x) + \eta \sum_{j=1}^{J} \rho_{jm} \, \mathbb{1}[x \in R_{jm}]$$

**Output:**

$$\boxed{F_M(x) = F_0(x) + \eta \sum_{m=1}^{M} \sum_{j=1}^{J} \rho_{jm} \, \mathbb{1}[x \in R_{jm}]}$$

---

## 3.6 Convergence Analysis

### Why does GBM converge?

**Claim:** At each step, the loss decreases (under mild conditions).

**Proof sketch:**

The update direction $$ h_m $$ is fitted to the negative gradient $$ -\nabla_F \mathcal{L} $$. Therefore:

$$\langle h_m, -\nabla_F \mathcal{L} \rangle = \sum_{i=1}^{N} h_m(x_i) \cdot r_{im} > 0$$

(positive inner product because $$ h_m $$ is correlated with $$ r_{im} $$ by construction).

This means $$ h_m $$ is a **descent direction**. With a sufficiently small $$ \eta $$, the loss decreases:

$$\mathcal{L}(F_m) < \mathcal{L}(F_{m-1})$$

This is the standard gradient descent convergence argument applied to function space.

---

## 3.7 GBM vs. XGBoost: The Mathematical Difference

| Aspect | GBM (Friedman) | XGBoost (Chen & Guestrin) |
|---|---|---|
| Gradient info | First-order only ($$ g_i $$) | First + Second-order ($$ g_i, h_i $$) |
| Step size | Line search (1D optimisation) | Closed-form via Hessian |
| Regularisation | Shrinkage + early stopping | Explicit $$ \gamma T + \frac{1}{2}\lambda\|w\|^2 $$ |
| Split criterion | Variance reduction on pseudo-residuals | $$ G^2/(H+\lambda) $$ gain formula |
| Equivalent to | Gradient descent | **Newton's method** |

---

### ❓ Why does Newton's method (XGBoost) converge faster than gradient descent (GBM)?

In classical optimisation:
- Gradient descent: linear convergence $$ \|\theta_t - \theta^*\| \leq c^t $$ where $$ c < 1 $$
- Newton's method: **quadratic** convergence $$ \|\theta_t - \theta^*\| \leq c \|\theta_{t-1} - \theta^*\|^2 $$

The Hessian rescales the gradient to account for curvature. In poorly-conditioned problems (different features have very different scales), Newton converges dramatically faster.

In boosting terms:
- GBM fits to residuals but doesn't know "how confident" to be in each leaf
- XGBoost knows the curvature at each leaf and calibrates the step optimally

---

### ❓ If XGBoost is strictly better, why study GBM at all?

1. **Conceptual foundation:** GBM provides the cleanest explanation of "boosting = gradient descent in function space"
2. **Historical importance:** GBM (2001) predates XGBoost (2016) by 15 years
3. **Robustness:** For some losses (e.g., quantile regression), the Hessian is degenerate or zero, and first-order methods are more stable
4. **Simpler implementation:** No need to compute or store second derivatives
5. **Understanding the hierarchy:** GBM is the bridge between "fit residuals" intuition and XGBoost's rigorous formulation

---
---
# 4. CatBoost (Categorical Boosting)

## 4.1 The Fundamental Problem: Target Leakage in Gradient Boosting

CatBoost (Prokhorenkova et al., 2018) identifies and solves a **subtle bias** that exists in ALL previous gradient boosting methods.

---

### The Prediction Shift Problem

In standard GBM/XGBoost, at iteration $$ t $$, we compute gradients:

$$g_i = \frac{\partial L(y_i, \hat{y})}{\partial \hat{y}} \Bigg|_{\hat{y} = F_{t-1}(x_i)}$$

Here $$ F_{t-1} $$ was trained on the **same data** $$ (x_i, y_i) $$ that we're now computing gradients for.

This means the gradients are **biased** — they reflect how well $$ F_{t-1} $$ fits the training data, not how well it predicts on fresh data.

---

### ❓ Why is this a problem? Isn't this just normal training?

Consider what happens:
1. $$ F_{t-1} $$ is fitted to $$ y_i $$, so $$ F_{t-1}(x_i) \approx y_i $$ (especially for overfit models)
2. The gradient $$ g_i \approx 0 $$ because the model already "knows" $$ y_i $$
3. The next tree $$ f_t $$ sees artificially small gradients → learns too little
4. On **test data**, $$ F_{t-1}(x_{\text{new}}) $$ is NOT close to $$ y_{\text{new}} $$, so the gradient would have been much larger

The result: the model's training-time gradient distribution is **shifted** compared to what it would see at test time. This is called **prediction shift**.

**Formally:**

$$\mathbb{E}[g_i \mid x_i] \neq \mathbb{E}[g \mid x] \text{ for a new point } x$$

The conditional distribution of gradients on training data is biased toward zero.

---

### ❓ How severe is this in practice?

- For **large datasets**: the effect is small (each point has negligible influence on $$ F_{t-1} $$)
- For **small datasets or high learning rate**: the shift is significant and causes overfitting
- For **many boosting rounds**: the bias accumulates across iterations

This is one reason why XGBoost/GBM overfit more on small datasets — not just variance, but actual bias in the gradient estimates.

## 4.2 Derivation: Ordered Boosting (The Key Innovation)

CatBoost's solution: train the model at each step using only **preceding** samples in a random permutation.

---

### Step 1: Random permutation

Draw a random permutation $$ \sigma $$ of $$ \{1, 2, \ldots, N\} $$.

Define the ordering: sample $$ \sigma(1) $$ comes first, $$ \sigma(2) $$ second, ..., $$ \sigma(N) $$ last.

---

### Step 2: Ordered model for each sample

For each sample $$ \sigma(i) $$, define a model $$ F_{t-1}^{(i)} $$ that is trained **only on the samples that appear before $$ \sigma(i) $$ in the permutation**:

$$F_{t-1}^{(i)} \text{ is trained on } \{(x_{\sigma(1)}, y_{\sigma(1)}), \ldots, (x_{\sigma(i-1)}, y_{\sigma(i-1)})\}$$

Crucially: $$ F_{t-1}^{(i)} $$ has **never seen** sample $$ \sigma(i) $$ during training.

---

### Step 3: Unbiased gradient computation

The gradient for sample $$ \sigma(i) $$ is now:

$$g_{\sigma(i)} = \frac{\partial L(y_{\sigma(i)}, \hat{y})}{\partial \hat{y}} \Bigg|_{\hat{y} = F_{t-1}^{(i)}(x_{\sigma(i)})}$$

Since $$ F_{t-1}^{(i)} $$ never saw $$ (x_{\sigma(i)}, y_{\sigma(i)}) $$:

$$\mathbb{E}[g_{\sigma(i)} \mid x_{\sigma(i)}] = \mathbb{E}[g \mid x] \text{ (unbiased!)}$$

---

### ❓ Why does a permutation fix the bias?

The bias arose because the model used to compute gradients had already memorised $$ y_i $$. By ensuring each sample's gradient is computed from a model that **excludes** that sample, we get the same distribution as test-time prediction.

This is analogous to **leave-one-out** estimation, but made computationally feasible through the permutation trick.

---

### ❓ Isn't this extremely expensive? (N different models!)

Yes, naively we'd need $$ N $$ separate models. CatBoost's trick:

1. **One permutation per boosting round** (not per sample)
2. Models are built **incrementally**: $$ F_{t-1}^{(i)} $$ is obtained by training on one more sample than $$ F_{t-1}^{(i-1)} $$
3. In practice, use a small number of random permutations (typically 1–4) and average

The complexity overhead is modest: roughly $$ O(\log N) $$ factor compared to standard boosting.

---

### Step 4: Formal guarantees

Let $$ \mathcal{F}_i = \sigma\{(x_{\sigma(1)}, y_{\sigma(1)}), \ldots, (x_{\sigma(i-1)}, y_{\sigma(i-1)})\} $$ be the filtration (information available to model $$ i $$).

Then $$ F_{t-1}^{(i)} $$ is $$ \mathcal{F}_i $$-measurable, and $$ (x_{\sigma(i)}, y_{\sigma(i)}) $$ is independent of $$ \mathcal{F}_i $$ (by the random permutation).

Therefore:

$$\mathbb{E}\left[g_{\sigma(i)} \mid \mathcal{F}_i, x_{\sigma(i)}\right] = \mathbb{E}\left[\frac{\partial L(y, \hat{y})}{\partial \hat{y}}\Bigg|_{\hat{y} = F_{t-1}^{(i)}(x)} \;\middle|\; x = x_{\sigma(i)}\right]$$

which is the **correct** conditional gradient, free of target leakage. ✓

## 4.3 Derivation: Ordered Target Statistics (Handling Categorical Features)

The second major innovation: a principled way to encode categorical features.

---

### The Problem with Standard Approaches

| Method | Issue |
|---|---|
| One-hot encoding | Exponential feature space for high-cardinality categoricals |
| Label encoding (1,2,3...) | Imposes false ordinal relationship |
| Target encoding ($$ \bar{y} $$ per category) | **Target leakage**: uses $$ y_i $$ to create features for the same $$ y_i $$ |

---

### ❓ Why is naive target encoding leaky?

Standard target statistic for category $$ k $$:

$$\hat{x}_i = \frac{\sum_{j: x_j = k} y_j}{|\{j: x_j = k\}|}$$

This includes $$ y_i $$ itself in the numerator! For rare categories (few samples), $$ \hat{x}_i \approx y_i $$, which is pure label leakage.

---

### CatBoost's Solution: Ordered Target Statistics (OTS)

Use the **same permutation** from ordered boosting. For sample $$ \sigma(i) $$ with categorical value $$ x_{\sigma(i)} = k $$:

$$\boxed{\hat{x}_{\sigma(i)} = \frac{\displaystyle\sum_{j < i:\, x_{\sigma(j)} = k} y_{\sigma(j)} + a \cdot p}{\displaystyle\left|\{j < i : x_{\sigma(j)} = k\}\right| + a}}$$

where:
- The sum is **only over preceding samples** in the permutation (never includes $$ y_{\sigma(i)} $$)
- $$ p $$ = prior (typically the global mean $$ \bar{y} $$)
- $$ a > 0 $$ = smoothing parameter (strength of the prior)

---

### Step-by-step construction

**Example:** Permutation order is $$ [\sigma(1), \sigma(2), \sigma(3), \sigma(4), \ldots] $$ and category feature is "colour":

| Position in perm | Sample | Colour | $$ y $$ | OTS value for "Red" |
|---|---|---|---|---|
| 1 | $$ \sigma(1) $$ | Red | 1 | $$ \frac{0 + a \cdot p}{0 + a} = p $$ (pure prior) |
| 2 | $$ \sigma(2) $$ | Blue | 0 | (not Red, irrelevant) |
| 3 | $$ \sigma(3) $$ | Red | 0 | $$ \frac{1 + ap}{1 + a} $$ |
| 4 | $$ \sigma(4) $$ | Red | 1 | $$ \frac{1+0 + ap}{2 + a} = \frac{1+ap}{2+a} $$ |

Each sample sees a **different** encoding based on what came before it. No sample ever sees its own label.

---

### ❓ Why the Bayesian prior term $$ a \cdot p $$?

Without smoothing ($$ a = 0 $$):
- First sample of category $$ k $$ has statistic $$ 0/0 $$ (undefined!)
- Second sample has statistic based on just one observation (extremely noisy)

With smoothing:
- The prior $$ p $$ provides a stable estimate when few samples are available
- As more samples accumulate, the data overwhelms the prior
- This is **Bayesian shrinkage**: $$ \hat{x} = \frac{n \cdot \bar{y}_k + a \cdot p}{n + a} $$

The parameter $$ a $$ controls the bias-variance tradeoff:
- Large $$ a $$: heavy shrinkage toward the prior, less variance, more bias
- Small $$ a $$: data dominates quickly, less bias, more variance

---

### ❓ How does this relate to the ordered boosting?

They use the **same permutation**! This ensures consistency:
1. The categorical encoding for sample $$ \sigma(i) $$ uses only predecessors
2. The gradient for sample $$ \sigma(i) $$ uses a model trained only on predecessors
3. Everything sample $$ \sigma(i) $$ sees is independent of its own label

This unified permutation scheme eliminates ALL forms of target leakage simultaneously.

## 4.4 Oblivious Decision Trees (Symmetric Trees)

CatBoost uses a unique tree structure: **oblivious trees** (also called symmetric trees).

---

### Definition

An oblivious tree of depth $$ d $$ uses the **same splitting condition at every node of the same level**.

Standard tree (asymmetric):
```
        [x1 > 5?]
       /         \
  [x2 > 3?]    [x3 > 7?]
   /    \        /    \
  L1    L2      L3    L4
```

Oblivious tree (symmetric):
```
        [x1 > 5?]         <- Level 0: same split everywhere
       /         \
  [x2 > 3?]    [x2 > 3?]  <- Level 1: same split everywhere
   /    \        /    \
  L1    L2      L3    L4
```

A depth-$$ d $$ oblivious tree has exactly $$ 2^d $$ leaves and $$ d $$ splits.

---

### Mathematical Representation

The tree output can be written as:

$$f(x) = \sum_{j=0}^{2^d - 1} w_j \cdot \prod_{l=0}^{d-1} c_l(x, j)$$

where:
- $$ w_j $$ is the leaf weight for leaf $$ j $$
- $$ c_l(x, j) $$ encodes whether the $$ l $$-th split condition sends $$ x $$ toward leaf $$ j $$:

$$c_l(x, j) = \begin{cases} \mathbb{1}[x_{f_l} > t_l] & \text{if bit } l \text{ of } j \text{ is 1} \\ \mathbb{1}[x_{f_l} \leq t_l] & \text{if bit } l \text{ of } j \text{ is 0} \end{cases}$$

The leaf index is determined by the binary string of split outcomes: each sample's path is a $$ d $$-bit number.

---

### ❓ Why oblivious trees? What's the mathematical advantage?

**1. Fast inference (CPU-friendly):**

Prediction requires evaluating $$ d $$ conditions and looking up a table:

$$\text{leaf\_index} = \sum_{l=0}^{d-1} 2^l \cdot \mathbb{1}[x_{f_l} > t_l]$$

This is a single index computation — no branching, no tree traversal. Extremely cache-friendly and SIMD-parallelisable.

**2. Natural regularisation:**

Fewer unique splits = fewer degrees of freedom:
- Standard depth-$$ d $$ tree: up to $$ 2^d - 1 $$ different splits
- Oblivious depth-$$ d $$ tree: exactly $$ d $$ different splits

This is a structural constraint that prevents overfitting.

**3. Balanced leaves:**

Every leaf has a roughly equal share of samples (since the same split applies uniformly). This prevents the "few-sample leaf" problem that plagues standard trees.

---

### ❓ Doesn't this limit expressiveness?

Yes! A single oblivious tree is less expressive than a standard tree of the same depth.

But CatBoost compensates by using **more trees** (larger ensemble). The total ensemble capacity is:

$$\text{Standard: } K \times (2^d - 1) \text{ unique splits}$$
$$\text{Oblivious: } K \times d \text{ unique splits per tree, but } K \text{ is larger}$$

Empirical finding: many weaker trees > fewer stronger trees (better generalisation, same reason as AdaBoost's weak learner philosophy).

## 4.5 The Complete CatBoost Algorithm

---

**Input:** Training data $$ \{(x_i, y_i)\}_{i=1}^N $$, loss $$ L $$, hyperparameters $$ (\eta, K, d, a, p) $$, categorical feature indices

**Step 0:** Generate $$ s $$ random permutations $$ \sigma_1, \ldots, \sigma_s $$

**Step 1: Compute Ordered Target Statistics** for all categorical features:

For each permutation $$ \sigma $$, each sample $$ \sigma(i) $$, each categorical feature with value $$ k $$:

$$\hat{x}_{\sigma(i)} = \frac{\sum_{j < i:\, x_{\sigma(j)} = k} y_{\sigma(j)} + a \cdot p}{|\{j < i : x_{\sigma(j)} = k\}| + a}$$

**For** $$ t = 1, 2, \ldots, K $$:

> 1. **Select a permutation** $$ \sigma $$ (cycle through or randomly choose)
>
> 2. **Compute ordered gradients:** For each sample $$ \sigma(i) $$:
>    - Use model $$ F_{t-1}^{(i)} $$ trained on $$ \{\sigma(1), \ldots, \sigma(i-1)\} $$
>    - Compute: $$ g_{\sigma(i)} = \frac{\partial L}{\partial \hat{y}}\Big|_{\hat{y} = F_{t-1}^{(i)}(x_{\sigma(i)})} $$
>
> 3. **Build an oblivious tree** of depth $$ d $$:
>    - For each level $$ l = 0, \ldots, d-1 $$:
>      - Find the best (feature, threshold) pair using the XGBoost gain formula:
>        $$\text{Gain} = \frac{1}{2}\left[\frac{G_L^2}{H_L+\lambda} + \frac{G_R^2}{H_R+\lambda} - \frac{(G_L+G_R)^2}{H_L+H_R+\lambda}\right] - \gamma$$
>      - Apply this **same split to all nodes** at this level
>
> 4. **Compute leaf weights** (same as XGBoost):
>    $$w_j^* = -\frac{G_j}{H_j + \lambda} \quad \text{for each leaf } j = 0, \ldots, 2^d - 1$$
>
> 5. **Update predictions:**
>    $$F_t(x) = F_{t-1}(x) + \eta \cdot f_t(x)$$

**Output:**

$$\boxed{F(x) = F_0 + \eta \sum_{t=1}^{K} f_t(x)}$$

---

## 4.6 Theoretical Analysis: Why CatBoost Reduces Overfitting

### Bias-Variance Decomposition

For standard gradient boosting, the gradient estimate has:

$$\text{Bias}[g_i] = \mathbb{E}[g_i] - g_i^{\text{true}} \neq 0 \quad \text{(prediction shift)}$$

$$\text{Var}[g_i] = \text{Var}[g_i \mid x_i]$$

For CatBoost's ordered gradients:

$$\text{Bias}[g_{\sigma(i)}] = 0 \quad \text{(unbiased by construction)}$$

$$\text{Var}[g_{\sigma(i)}] = \text{Var}[g \mid x] + \underbrace{\frac{1}{i-1}\text{estimation noise}}_{\text{from using fewer samples}}$$

The tradeoff: CatBoost eliminates bias at the cost of slightly higher variance (since $$ F_{t-1}^{(i)} $$ uses fewer training samples). But the variance penalty is $$ O(1/N) $$ while the bias fix is $$ O(1) $$, so the net effect is beneficial.

---

### ❓ How many permutations should we use?

- **1 permutation**: Fastest, but high variance in gradient estimates
- **4 permutations** (default): Good bias-variance tradeoff
- **More permutations**: Diminishing returns; variance decreases as $$ O(1/\sqrt{s}) $$

The CatBoost paper shows that even 1 permutation eliminates the bulk of prediction shift, and performance stabilises by $$ s = 4 $$.

## 4.7 The Full Picture: All Four Algorithms Compared

---

### Mathematical Foundation Comparison

| | AdaBoost | GBM | XGBoost | CatBoost |
|---|---|---|---|---|
| **Year** | 1995 | 2001 | 2016 | 2018 |
| **Objective** | $$ \sum e^{-y_iF(x_i)} $$ | $$ \sum L(y_i, F(x_i)) $$ | $$ \sum L + \Omega(f) $$ | $$ \sum L + \Omega(f) $$ |
| **Gradient order** | Exact (exp. loss) | 1st order | 2nd order | 2nd order |
| **Regularisation** | Implicit (weak learners) | $$ \eta $$ only | $$ \gamma T + \frac{1}{2}\lambda\|w\|^2 $$ | Same + ordered boosting |
| **Split criterion** | Weighted error | Variance of pseudo-residuals | $$ G^2/(H+\lambda) $$ gain | Same gain, oblivious structure |
| **Gradient estimation** | Uses all data | Uses all data | Uses all data | **Ordered** (leak-free) |
| **Categoricals** | One-hot | One-hot | One-hot | **Ordered target stats** |
| **Tree type** | Stumps | Standard | Standard | **Oblivious (symmetric)** |
| **Convergence type** | Exponential train error | Linear (GD) | Quadratic (Newton) | Quadratic (Newton) |

---

### The Evolution of Key Ideas

$$\text{AdaBoost} \xrightarrow{\text{generalise loss}} \text{GBM} \xrightarrow{\text{add Hessian + regularise}} \text{XGBoost} \xrightarrow{\text{fix leakage + categoricals}} \text{CatBoost}$$

Each step adds one fundamental improvement:
1. **AdaBoost → GBM**: Any loss function (functional gradient descent)
2. **GBM → XGBoost**: Second-order info + explicit complexity control
3. **XGBoost → CatBoost**: Unbiased gradients + native categorical handling

---

### ❓ When to use which?

| Scenario | Best choice | Why |
|---|---|---|
| Small dataset, many categoricals | **CatBoost** | Ordered boosting prevents overfit; native categoricals |
| Large dataset, need speed | **XGBoost/LightGBM** | Histogram methods scale; leakage is less of an issue |
| Interpretability needed | **AdaBoost + stumps** | Each stump is a simple rule |
| Custom/exotic loss | **GBM** | Clean framework; just supply the gradient |
| Production inference latency matters | **CatBoost** | Oblivious trees are fastest at prediction time |
| Need a baseline | **XGBoost** | Best documented, most robust defaults |

---
---
# 5. Frequently Asked "Out-of-Box" Exam Questions

## Q1: "Can AdaBoost overfit?"

**Yes**, but it's resistant. Overfit happens when:
- Data is noisy (mislabelled points get exponentially upweighted → model chases noise)
- Too many rounds on separable data (margins grow but at diminishing returns)
- Base learners are too complex (deep trees instead of stumps)

Exponential loss is **not robust to outliers** — a misclassified point with huge weight dominates.

---

## Q2: "What happens if we use log-loss instead of exponential loss in the AdaBoost framework?"

You get **LogitBoost** (Friedman et al., 2000). The weight update becomes smoother:
- Exponential loss: weights grow as $$ e^{\alpha} $$ (explosive for hard examples)
- Log-loss: weights grow as $$ 1/p_i $$ (bounded, more robust)

LogitBoost is more robust to noise but loses the elegant closed-form of AdaBoost.

---

## Q3: "Why does XGBoost use $$ G_j^2/(H_j + \lambda) $$ instead of something like information gain or Gini?"

Because $$ G_j^2/(H_j+\lambda) $$ is **derived from the objective**, not chosen heuristically:
- Gini/entropy are heuristics for classification trees
- $$ G_j^2/(H_j+\lambda) $$ directly measures "how much can the loss decrease in this region"
- It automatically adapts to any loss function (regression, classification, ranking)
- The $$ \lambda $$ provides principled regularisation, unlike ad-hoc pruning

---

## Q4: "What if $$ H_j + \lambda \approx 0 $$ for some leaf?"

Then $$ w_j^* = -G_j / (H_j + \lambda) $$ could be extremely large.

**Why this is dangerous:** Few samples in a leaf with flat loss → no curvature information → model makes wild predictions.

**Solutions (all used in XGBoost):**
1. $$ \lambda > 0 $$ ensures the denominator is always positive
2. `min_child_weight` parameter: don't split if $$ H_j < $$ threshold
3. Learning rate $$ \eta $$ shrinks the leaf value anyway

This is exactly why $$ \lambda $$ exists — it's a safeguard against degenerate cases.

---

## Q5: "Why does GBM fit to pseudo-residuals using least squares, even for log-loss?"

The pseudo-residuals $ r_{im} $ are a **vector in $ \mathbb{R}^N $** representing the descent direction. We need a function $ h(x) $ that approximates this vector at all $ N $ points and can generalise to new $ x $.

Least squares is the **natural projection** of the gradient vector onto the function class $ \mathcal{H} $:

$h_m = \arg\min_h \sum_i \|r_{im} - h(x_i)\|^2 = \text{orthogonal projection of } \mathbf{r} \text{ onto } \mathcal{H}$

This guarantees the fitted function has **positive inner product** with the gradient (i.e., is a descent direction), regardless of the original loss. The actual step size is then corrected by line search or learning rate.

---

## Q6: "Explain the prediction shift problem with a concrete example."

Suppose $ N = 10 $ and we've run 100 rounds of XGBoost with depth-6 trees:
- Training error: 0.01 (model has nearly memorised the training data)
- $ F_{100}(x_i) \approx y_i $ for all training points
- Gradients: $ g_i \approx 0 $ (model thinks there's nothing left to learn)

But on a new test point with similar $ x $:
- $ F_{100}(x_{\text{new}}) $ might be off by a lot (generalisation gap)
- The "true" gradient for this region is NOT near zero

The model stops improving because it's computing gradients on data it has already memorised. CatBoost fixes this by computing each gradient from a model that hasn't seen that point.

---

## Q7: "Can we combine ordered boosting with standard (non-oblivious) trees?"

Yes! Ordered boosting and oblivious trees are **independent** innovations:
- Ordered boosting = how gradients are estimated (fixes bias)
- Oblivious trees = tree structure (fast inference + regularisation)

CatBoost uses both together, but you could apply ordered boosting with standard trees. The paper shows ordered boosting alone gives most of the improvement; oblivious trees add inference speed and modest accuracy gains.

---

## Q8: "Derive: what happens to CatBoost's ordered target statistic as $ N \to \infty $?"

For a category $ k $ with true mean $ \mu_k = \mathbb{E}[y \mid x = k] $:

$\hat{x}_{\sigma(i)} = \frac{\sum_{j<i: x_{\sigma(j)}=k} y_{\sigma(j)} + ap}{|\{j<i: x_{\sigma(j)}=k\}| + a}$

As $ i \to \infty $, the number of predecessors with category $ k $ grows as $ n_k \sim i \cdot P(x=k) $:

$\hat{x}_{\sigma(i)} \to \frac{n_k \cdot \mu_k + ap}{n_k + a} \to \mu_k \quad \text{as } n_k \to \infty$

So the ordered target statistic converges to the **true conditional mean** — the optimal encoding for regression. The prior $ ap $ becomes irrelevant for large $ N $. ✓

---

## Q6: "Derive the connection: if loss is squared error, what does the XGBoost gain formula reduce to?"

For $$ L = \frac{1}{2}(y_i - \hat{y}_i)^2 $$:
- $$ g_i = \hat{y}_i^{(t-1)} - y_i = -r_i $$ (negative residual)
- $$ h_i = 1 $$

So $$ G_j = -\sum_{i \in I_j} r_i $$ and $$ H_j = |I_j| = n_j $$.

$$\text{Gain} = \frac{1}{2}\left[\frac{(\sum_{i \in I_L} r_i)^2}{n_L + \lambda} + \frac{(\sum_{i \in I_R} r_i)^2}{n_R + \lambda} - \frac{(\sum_{i \in I} r_i)^2}{n + \lambda}\right] - \gamma$$

With $$ \lambda=0, \gamma=0 $$, this becomes the classical **variance reduction** criterion:

$$\text{Gain} \propto n_L \cdot \bar{r}_L^2 + n_R \cdot \bar{r}_R^2 - n \cdot \bar{r}^2$$

which is exactly how CART builds regression trees on residuals.